<a href="https://colab.research.google.com/github/Otza02/land2vec/blob/main/notebooks/prueba_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!git clone https://github.com/Otza02/land2vec.git
%cd /content/land2vec

Cloning into 'land2vec'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 93 (delta 35), reused 76 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 4.01 MiB | 10.25 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/land2vec


In [ ]:
!pip install --no-cache-dir -e .

Obtaining file:///content/land2vec
  Preparing metadata (setup.py) ... done
  Running setup.py develop for land2vec


In [1]:
import tqdm

import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import lr_scheduler

from land2vec.dataset import load_data, SequenceDataset
from land2vec.config import Config
from land2vec.model import DecoderTransformer, train_loop, val_loop
from land2vec.tokenizer import Tokenizer

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
config = Config()

In [7]:
print("loading data")
file_path = "data/id_seqs_text_2000_2022_chaco_santiago_frontier.zip"
dataset = load_data(file_path="/content/land2vec/" + file_path, window=8)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
print("spliting data")
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False
)

loading data


100%|██████████| 1424457/1424457 [01:50<00:00, 12868.31it/s]


spliting data


In [8]:
config

Config(block_size=8, n_embd=32, n_head=2, n_layer=2, dropout=0.1, epochs=25, patience=5, batch_size=128, lr=0.001, min_lr=1e-06, weight_decay=0.01)

In [ ]:
model = DecoderTransformer(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
    dropout=config.dropout
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=config.min_lr)

train_losses = []
val_losses = []

best_val_loss = float("inf")
patience_counter = 0

for epoch in range(config.epochs):
    train_loss = train_loop(model, train_loader, optimizer, device)
    val_loss = val_loop(model, val_loader, device)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1
    current_lr = scheduler.get_last_lr()[0]

    print(f"Epoch {epoch+1} \nlr={current_lr:.6f}")

    if patience_counter >= config.patience:
        print("Early stopping triggered")
        break

model.load_state_dict(torch.load("best_model.pt"))

ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
torch.save(model.state_dict(), "first-test.pt")

In [ ]:
model = DecoderTransformer(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
).to(device)

model.load_state_dict(torch.load("models/first-test.pt"))

model.eval()

GPT(
  (token_embedding): Embedding(16, 32)
  (position_embedding): Embedding(8, 32)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=32, out_features=16, bias=True)
)

In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in tqdm.tqdm(test_loader):
        x = x.long().to(device)
        y = y.long().to(device)

        logits, loss = model(x, y)

        preds = torch.argmax(logits, dim=-1)

        correct += (preds == y).sum().item()
        total += y.numel()

        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

accuracy = correct / total

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.9953
